# Neural-Network Regression with 10,000-Epoch Training

This notebook evaluates one-hidden-layer neural networks across classical and overparameterised capacity regimes. Each architecture is trained for a fixed budget of 10,000 complete epochs without early stopping. The checkpoint with the minimum validation loss is selected retrospectively after training.

In [ ]:
Import all libraries and check the computing device

from pathlib import Path
from IPython.display import display

import copy
import gc
import json
import math
import random
import time
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from sklearn.feature_selection import mutual_info_regression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.preprocessing import StandardScaler

from torch import nn
from torch.utils.data import (
    DataLoader,
    TensorDataset
)

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

random_seed = 42

random.seed(random_seed)
np.random.seed(random_seed)
torch.manual_seed(random_seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(random_seed)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("PyTorch version:", torch.__version__)
print("Computing device:", device)

if torch.cuda.is_available():
    print(
        "GPU model:",
        torch.cuda.get_device_name(0)
    )

print("Random seed:", random_seed)

In [ ]:
# Load training and validation data and seed 84 descriptors

model_data_file = Path(
    "model_dataset.csv"
)

training_feature_file = Path(
    "mordred_training_features_filtered.pkl"
)

validation_feature_file = Path(
    "mordred_validation_features_filtered.pkl"
)

seed_84_feature_file = Path(
    "full_ga_results_seed_84/"
    "full_ga_best_features.csv"
)

required_files = [
    model_data_file,
    training_feature_file,
    validation_feature_file,
    seed_84_feature_file
]

missing_files = [
    str(file)
    for file in required_files
    if not file.exists()
]

if missing_files:
    raise FileNotFoundError(
        f"Missing files: {missing_files}"
    )

# Load the filtered Mordred descriptor matrices
X_train_all = pd.read_pickle(
    training_feature_file
).astype("float32")

X_valid_all = pd.read_pickle(
    validation_feature_file
).astype("float32")

# Load continuous docking-score targets
model_data = pd.read_csv(
    model_data_file
)

unnamed_columns = [
    column
    for column in model_data.columns
    if column.startswith("Unnamed")
]

if len(unnamed_columns) == 1:
    saved_index_column = unnamed_columns[0]

    if model_data[
        saved_index_column
    ].is_unique:
        model_data = model_data.set_index(
            saved_index_column
        )
    else:
        model_data = model_data.drop(
            columns=unnamed_columns
        )

# Load the descriptors selected by GA seed 84
seed_84_feature_data = pd.read_csv(
    seed_84_feature_file
)

seed_84_feature_data = (
    seed_84_feature_data.loc[
        :,
        ~seed_84_feature_data.columns.str.startswith(
            "Unnamed"
        )
    ]
)

if "descriptor" in seed_84_feature_data.columns:
    descriptor_column = "descriptor"
else:
    descriptor_column = (
        seed_84_feature_data.columns[0]
    )

seed_84_descriptors = (
    seed_84_feature_data[
        descriptor_column
    ]
    .dropna()
    .astype(str)
    .tolist()
)

target_columns = [
    "jnk3_score",
    "gsk3b_score"
]

# Check targets and descriptors
missing_targets = [
    target
    for target in target_columns
    if target not in model_data.columns
]

if missing_targets:
    raise ValueError(
        f"Missing targets: {missing_targets}"
    )

missing_training_descriptors = [
    descriptor
    for descriptor in seed_84_descriptors
    if descriptor not in X_train_all.columns
]

missing_validation_descriptors = [
    descriptor
    for descriptor in seed_84_descriptors
    if descriptor not in X_valid_all.columns
]

if missing_training_descriptors:
    raise ValueError(
        "Seed 84 descriptors missing from "
        f"training data: {missing_training_descriptors[:10]}"
    )

if missing_validation_descriptors:
    raise ValueError(
        "Seed 84 descriptors missing from "
        f"validation data: {missing_validation_descriptors[:10]}"
    )

# Confirm that feature indices match the target data
if not X_train_all.index.isin(
    model_data.index
).all():
    raise ValueError(
        "Training feature indices do not match "
        "model_dataset.csv."
    )

if not X_valid_all.index.isin(
    model_data.index
).all():
    raise ValueError(
        "Validation feature indices do not match "
        "model_dataset.csv."
    )

# Keep the seed 84 descriptor subset
X_train_seed84 = X_train_all[
    seed_84_descriptors
].copy()

X_valid_seed84 = X_valid_all[
    seed_84_descriptors
].copy()

# Align the two continuous regression targets
y_train = model_data.loc[
    X_train_seed84.index,
    target_columns
].astype("float32")

y_valid = model_data.loc[
    X_valid_seed84.index,
    target_columns
].astype("float32")

# Final checks
if X_train_seed84.isna().any().any():
    raise ValueError(
        "Missing values found in training features."
    )

if X_valid_seed84.isna().any().any():
    raise ValueError(
        "Missing values found in validation features."
    )

if y_train.isna().any().any():
    raise ValueError(
        "Missing values found in training targets."
    )

if y_valid.isna().any().any():
    raise ValueError(
        "Missing values found in validation targets."
    )

print(
    "Full training Mordred shape:",
    X_train_all.shape
)

print(
    "Full validation Mordred shape:",
    X_valid_all.shape
)

print(
    "\nSeed 84 selected descriptors:",
    len(seed_84_descriptors)
)

print(
    "Seed 84 training shape:",
    X_train_seed84.shape
)

print(
    "Seed 84 validation shape:",
    X_valid_seed84.shape
)

print(
    "\nTraining target shape:",
    y_train.shape
)

print(
    "Validation target shape:",
    y_valid.shape
)

print(
    "Regression target order:",
    target_columns
)

print(
    "\nFeature order matches:",
    X_train_seed84.columns.equals(
        X_valid_seed84.columns
    )
)

print("Test set loaded:", False)

In [ ]:
#Reduce the seed 84 descriptors to 100 using training data only

number_of_reduced_descriptors = 100

descriptor_ranking = pd.DataFrame({
    "descriptor": seed_84_descriptors
})

for target in target_columns:

    mutual_information_scores = (
        mutual_info_regression(
            X_train_seed84,
            y_train[target],
            random_state=random_seed,
            n_jobs=-1
        )
    )

    score_column = (
        f"{target}_mutual_information"
    )

    rank_column = (
        f"{target}_rank"
    )

    descriptor_ranking[
        score_column
    ] = mutual_information_scores

    descriptor_ranking[
        rank_column
    ] = (
        descriptor_ranking[
            score_column
        ]
        .rank(
            ascending=False,
            method="average"
        )
    )

# Combine rankings from both regression targets
descriptor_ranking[
    "mean_rank"
] = (
    descriptor_ranking[
        [
            "jnk3_score_rank",
            "gsk3b_score_rank"
        ]
    ]
    .mean(axis=1)
)

descriptor_ranking = (
    descriptor_ranking
    .sort_values(
        [
            "mean_rank",
            "descriptor"
        ],
        ascending=[
            True,
            True
        ]
    )
    .reset_index(drop=True)
)

reduced_descriptors = (
    descriptor_ranking
    .head(
        number_of_reduced_descriptors
    )["descriptor"]
    .tolist()
)

X_train_reduced = (
    X_train_seed84[
        reduced_descriptors
    ]
    .copy()
)

X_valid_reduced = (
    X_valid_seed84[
        reduced_descriptors
    ]
    .copy()
)

descriptor_ranking.to_csv(
    "seed84_descriptor_mutual_information_ranking.csv",
    index=False
)

pd.DataFrame({
    "descriptor": reduced_descriptors
}).to_csv(
    "seed84_top100_descriptors.csv",
    index=False
)

print(
    "Original seed 84 descriptors:",
    len(seed_84_descriptors)
)

print(
    "Reduced descriptors:",
    len(reduced_descriptors)
)

print(
    "\nReduced training shape:",
    X_train_reduced.shape
)

print(
    "Reduced validation shape:",
    X_valid_reduced.shape
)

display(
    descriptor_ranking.head(20)
)

print("\nSaved:")
print(
    "seed84_descriptor_mutual_information_ranking.csv"
)
print(
    "seed84_top100_descriptors.csv"
)

In [ ]:
# Standardise features and targets using training data only

batch_size = 128
maximum_epochs = 10_000

feature_scaler = StandardScaler()
target_scaler = StandardScaler()

# Fit both scalers using training data only
X_train_scaled = feature_scaler.fit_transform(
    X_train_reduced
).astype("float32")

X_valid_scaled = feature_scaler.transform(
    X_valid_reduced
).astype("float32")

y_train_scaled = target_scaler.fit_transform(
    y_train
).astype("float32")

y_valid_scaled = target_scaler.transform(
    y_valid
).astype("float32")

# Convert the arrays to PyTorch tensors
X_train_tensor = torch.from_numpy(
    X_train_scaled
)

X_valid_tensor = torch.from_numpy(
    X_valid_scaled
)

y_train_tensor = torch.from_numpy(
    y_train_scaled
)

y_valid_tensor = torch.from_numpy(
    y_valid_scaled
)

training_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

# The final incomplete batch will be retained
batches_per_epoch = math.ceil(
    len(training_dataset) / batch_size
)

total_optimisation_steps = (
    maximum_epochs
    * batches_per_epoch
)

# Save the fitted transformations
joblib.dump(
    feature_scaler,
    "nn_10000_epochs_feature_scaler.pkl"
)

joblib.dump(
    target_scaler,
    "nn_10000_epochs_target_scaler.pkl"
)

print(
    "Training feature shape:",
    X_train_tensor.shape
)

print(
    "Validation feature shape:",
    X_valid_tensor.shape
)

print(
    "Training target shape:",
    y_train_tensor.shape
)

print(
    "Validation target shape:",
    y_valid_tensor.shape
)

print("\nBatch size:", batch_size)
print("Complete training epochs:", maximum_epochs)
print("Batches per complete epoch:", batches_per_epoch)

print(
    "Total planned optimisation steps:",
    f"{total_optimisation_steps:,}"
)

print(
    "\nTraining feature mean:",
    round(
        float(X_train_scaled.mean()),
        6
    )
)

print(
    "Training feature standard deviation:",
    round(
        float(X_train_scaled.std()),
        6
    )
)

print("\nSaved:")
print("nn_10000_epochs_feature_scaler.pkl")
print("nn_10000_epochs_target_scaler.pkl")

In [ ]:
# Define one-hidden-layer models and calculate parameter counts

class OneHiddenLayerRegressor(nn.Module):

    def __init__(
        self,
        input_size,
        hidden_size,
        output_size=2
    ):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(
                input_size,
                hidden_size
            ),
            nn.ReLU(),
            nn.Linear(
                hidden_size,
                output_size
            )
        )

    def forward(self, x):
        return self.network(x)


input_size = X_train_tensor.shape[1]
output_size = y_train_tensor.shape[1]
number_of_training_samples = len(
    X_train_tensor
)

hidden_sizes = [
    10,
    50,
    100,
    250,
    500,
    1000,
    2000
]

parameter_rows = []

for hidden_size in hidden_sizes:

    model = OneHiddenLayerRegressor(
        input_size=input_size,
        hidden_size=hidden_size,
        output_size=output_size
    )

    trainable_parameters = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )

    parameter_ratio = (
        trainable_parameters
        / number_of_training_samples
    )

    if parameter_ratio < 1:
        capacity_regime = "Classical"
    elif parameter_ratio < 2:
        capacity_regime = "Intermediate"
    else:
        capacity_regime = "Modern overparameterised"

    parameter_rows.append({
        "hidden_neurons": hidden_size,
        "trainable_parameters":
            trainable_parameters,
        "training_samples":
            number_of_training_samples,
        "parameters_per_training_sample":
            parameter_ratio,
        "capacity_regime":
            capacity_regime
    })

parameter_comparison = pd.DataFrame(
    parameter_rows
)

parameter_comparison.to_csv(
    "nn_10000_epochs_parameter_counts.csv",
    index=False
)

display(parameter_comparison)

print(
    "\nTwo times training samples:",
    2 * number_of_training_samples
)

print(
    "Three times training samples:",
    3 * number_of_training_samples
)

print(
    "\nSaved: nn_10000_epochs_parameter_counts.csv"
)

In [ ]:
# Define resumable fixed-epoch training functions

results_directory = Path(
    "nn_10000_epochs_results"
)

results_directory.mkdir(
    exist_ok=True
)

learning_rate = 0.001
checkpoint_interval = 10


def create_training_loader(
    data_generator
):

    return DataLoader(
        training_dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=False,
        num_workers=0,
        pin_memory=True,
        generator=data_generator
    )


@torch.no_grad()
def calculate_validation_loss(
    model,
    validation_features,
    validation_targets,
    loss_function
):

    model.eval()

    predictions = model(
        validation_features
    )

    validation_loss = loss_function(
        predictions,
        validation_targets
    )

    return float(
        validation_loss.item()
    )


def train_complete_epochs(
    hidden_size,
    seed=random_seed
):

    model_directory = (
        results_directory
        / f"hidden_{hidden_size}"
    )

    model_directory.mkdir(
        exist_ok=True
    )

    best_model_file = (
        model_directory
        / "best_model.pt"
    )

    resume_file = (
        model_directory
        / "resume_checkpoint.pt"
    )

    history_file = (
        model_directory
        / "training_history.csv"
    )

    summary_file = (
        model_directory
        / "training_summary.csv"
    )

    completed_file = (
        model_directory
        / "training_complete.json"
    )

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    model = OneHiddenLayerRegressor(
        input_size=input_size,
        hidden_size=hidden_size,
        output_size=output_size
    ).to(device)

    trainable_parameters = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )

    loss_function = nn.MSELoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate
    )

    data_generator = torch.Generator()
    data_generator.manual_seed(seed)

    start_epoch = 0
    best_epoch = 0
    best_validation_loss = np.inf
    history_records = []
    elapsed_seconds_before_resume = 0.0

    # Continue from the latest saved epoch
    if resume_file.exists():

        checkpoint = torch.load(
            resume_file,
            map_location=device,
            weights_only=False
        )

        model.load_state_dict(
            checkpoint["model_state_dict"]
        )

        optimizer.load_state_dict(
            checkpoint["optimizer_state_dict"]
        )

        start_epoch = int(
            checkpoint["completed_epoch"]
        )

        best_epoch = int(
            checkpoint["best_epoch"]
        )

        best_validation_loss = float(
            checkpoint[
                "best_validation_loss"
            ]
        )

        elapsed_seconds_before_resume = float(
            checkpoint.get(
                "elapsed_seconds",
                0.0
            )
        )

        if (
            "data_generator_state"
            in checkpoint
        ):
            data_generator.set_state(
                checkpoint[
                    "data_generator_state"
                ]
            )

        if history_file.exists():
            history_records = (
                pd.read_csv(
                    history_file
                )
                .to_dict("records")
            )

        print(
            f"Resuming hidden size "
            f"{hidden_size} from "
            f"epoch {start_epoch:,}."
        )

    if start_epoch >= maximum_epochs:

        print(
            f"Hidden size {hidden_size} "
            "is already complete."
        )

        return pd.read_csv(
            summary_file
        ).iloc[0].to_dict()

    training_loader = (
        create_training_loader(
            data_generator
        )
    )

    validation_features_device = (
        X_valid_tensor.to(
            device,
            non_blocking=True
        )
    )

    validation_targets_device = (
        y_valid_tensor.to(
            device,
            non_blocking=True
        )
    )

    run_start_time = time.time()

    progress_bar = tqdm(
        total=maximum_epochs,
        initial=start_epoch,
        desc=f"Hidden size {hidden_size}"
    )

    for epoch in range(
        start_epoch + 1,
        maximum_epochs + 1
    ):

        model.train()

        total_training_loss = 0.0
        total_training_samples = 0

        for (
            batch_features,
            batch_targets
        ) in training_loader:

            batch_features = (
                batch_features.to(
                    device,
                    non_blocking=True
                )
            )

            batch_targets = (
                batch_targets.to(
                    device,
                    non_blocking=True
                )
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            predictions = model(
                batch_features
            )

            training_loss = loss_function(
                predictions,
                batch_targets
            )

            training_loss.backward()
            optimizer.step()

            current_batch_size = (
                batch_features.shape[0]
            )

            total_training_loss += (
                training_loss.item()
                * current_batch_size
            )

            total_training_samples += (
                current_batch_size
            )

        mean_training_loss = (
            total_training_loss
            / total_training_samples
        )

        validation_loss = (
            calculate_validation_loss(
                model,
                validation_features_device,
                validation_targets_device,
                loss_function
            )
        )

        history_records.append({
            "epoch": epoch,
            "training_loss":
                mean_training_loss,
            "validation_loss":
                validation_loss
        })

        # Save the model with minimum validation loss
        if (
            validation_loss
            < best_validation_loss
        ):

            best_validation_loss = (
                validation_loss
            )

            best_epoch = epoch

            torch.save(
                {
                    "hidden_size":
                        hidden_size,
                    "input_size":
                        input_size,
                    "output_size":
                        output_size,
                    "trainable_parameters":
                        trainable_parameters,
                    "best_epoch":
                        best_epoch,
                    "best_validation_loss":
                        best_validation_loss,
                    "model_state_dict":
                        {
                            name:
                                value.detach()
                                .cpu()
                                .clone()
                            for name, value
                            in model.state_dict().items()
                        }
                },
                best_model_file
            )

        # Save progress regularly for resuming
        if (
            epoch % checkpoint_interval == 0
            or epoch == maximum_epochs
        ):

            current_elapsed_seconds = (
                elapsed_seconds_before_resume
                + time.time()
                - run_start_time
            )

            pd.DataFrame(
                history_records
            ).to_csv(
                history_file,
                index=False
            )

            torch.save(
                {
                    "hidden_size":
                        hidden_size,
                    "completed_epoch":
                        epoch,
                    "best_epoch":
                        best_epoch,
                    "best_validation_loss":
                        best_validation_loss,
                    "model_state_dict":
                        model.state_dict(),
                    "optimizer_state_dict":
                        optimizer.state_dict(),
                    "data_generator_state":
                        data_generator.get_state(),
                    "elapsed_seconds":
                        current_elapsed_seconds
                },
                resume_file
            )

        progress_bar.update(1)

        if (
            epoch % 10 == 0
            or epoch == maximum_epochs
        ):

            progress_bar.set_postfix({
                "valid_loss":
                    f"{validation_loss:.6f}",
                "best_epoch":
                    best_epoch
            })

    progress_bar.close()

    total_elapsed_seconds = (
        elapsed_seconds_before_resume
        + time.time()
        - run_start_time
    )

    final_validation_loss = float(
        history_records[-1][
            "validation_loss"
        ]
    )

    model_summary = pd.DataFrame([{
        "hidden_neurons":
            hidden_size,
        "trainable_parameters":
            trainable_parameters,
        "completed_epochs":
            maximum_epochs,
        "optimisation_steps":
            maximum_epochs
            * batches_per_epoch,
        "best_epoch":
            best_epoch,
        "best_validation_loss":
            best_validation_loss,
        "final_validation_loss":
            final_validation_loss,
        "elapsed_hours":
            total_elapsed_seconds
            / 3600,
        "best_model_file":
            str(best_model_file),
        "history_file":
            str(history_file)
    }])

    model_summary.to_csv(
        summary_file,
        index=False
    )

    with open(
        completed_file,
        "w"
    ) as file:

        json.dump(
            {
                "hidden_size":
                    hidden_size,
                "completed_epochs":
                    maximum_epochs,
                "best_epoch":
                    best_epoch,
                "best_validation_loss":
                    best_validation_loss
            },
            file,
            indent=2
        )

    del model
    del optimizer
    del validation_features_device
    del validation_targets_device

    gc.collect()
    torch.cuda.empty_cache()

    return model_summary.iloc[0].to_dict()


print("Training functions prepared.")
print("Complete epochs per model:", maximum_epochs)
print("Batches per epoch:", batches_per_epoch)

print(
    "Optimisation steps per model:",
    f"{total_optimisation_steps:,}"
)

print(
    "Resume checkpoint interval:",
    checkpoint_interval,
    "epochs"
)

print("Early stopping used:", False)
print("Dropout used:", False)
print("Results directory:", results_directory)


In [ ]:
# Train all required architectures for 10,000 complete epochs

architecture_summaries = []

for hidden_size in hidden_sizes:

    print(
        f"\nStarting hidden size "
        f"{hidden_size}..."
    )

    model_summary = train_complete_epochs(
        hidden_size=hidden_size,
        seed=random_seed
    )

    architecture_summaries.append(
        model_summary
    )

    current_summary = (
        pd.DataFrame(
            architecture_summaries
        )
        .sort_values(
            "hidden_neurons"
        )
        .reset_index(drop=True)
    )

    current_summary.to_csv(
        results_directory
        / "architecture_training_summary.csv",
        index=False
    )

architecture_training_summary = (
    pd.DataFrame(
        architecture_summaries
    )
    .sort_values(
        "hidden_neurons"
    )
    .reset_index(drop=True)
)

architecture_training_summary.to_csv(
    results_directory
    / "architecture_training_summary.csv",
    index=False
)

display(
    architecture_training_summary[
        [
            "hidden_neurons",
            "trainable_parameters",
            "completed_epochs",
            "optimisation_steps",
            "best_epoch",
            "best_validation_loss",
            "final_validation_loss",
            "elapsed_hours"
        ]
    ]
)

print(
    "\nCompleted architectures:",
    len(architecture_training_summary)
)

print(
    "Saved:",
    results_directory
    / "architecture_training_summary.csv"
)

In [ ]:
# Evaluate the best validation checkpoint from each architecture

@torch.no_grad()
def predict_in_batches(
    model,
    feature_tensor,
    prediction_batch_size=4096
):

    prediction_dataset = TensorDataset(
        feature_tensor
    )

    prediction_loader = DataLoader(
        prediction_dataset,
        batch_size=prediction_batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=True
    )

    predictions = []

    model.eval()

    for (batch_features,) in prediction_loader:

        batch_features = batch_features.to(
            device,
            non_blocking=True
        )

        batch_predictions = model(
            batch_features
        )

        predictions.append(
            batch_predictions.cpu().numpy()
        )

    return np.concatenate(
        predictions,
        axis=0
    )


true_train_targets = y_train[
    target_columns
].to_numpy()

true_valid_targets = y_valid[
    target_columns
].to_numpy()

architecture_evaluation_rows = []

for hidden_size in hidden_sizes:

    model_directory = (
        results_directory
        / f"hidden_{hidden_size}"
    )

    best_model_file = (
        model_directory
        / "best_model.pt"
    )

    if not best_model_file.exists():
        raise FileNotFoundError(
            f"Missing checkpoint: {best_model_file}"
        )

    checkpoint = torch.load(
        best_model_file,
        map_location="cpu",
        weights_only=False
    )

    model = OneHiddenLayerRegressor(
        input_size=input_size,
        hidden_size=hidden_size,
        output_size=output_size
    ).to(device)

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    train_predictions_scaled = (
        predict_in_batches(
            model,
            X_train_tensor
        )
    )

    valid_predictions_scaled = (
        predict_in_batches(
            model,
            X_valid_tensor
        )
    )

    train_predictions = (
        target_scaler.inverse_transform(
            train_predictions_scaled
        )
    )

    valid_predictions = (
        target_scaler.inverse_transform(
            valid_predictions_scaled
        )
    )

    result = {
        "hidden_neurons": hidden_size,
        "trainable_parameters":
            checkpoint[
                "trainable_parameters"
            ],
        "best_epoch":
            checkpoint["best_epoch"],
        "best_validation_loss":
            checkpoint[
                "best_validation_loss"
            ]
    }

    valid_rmse_values = []
    valid_r2_values = []

    for target_index, target in enumerate(
        target_columns
    ):

        result[
            f"{target}_train_mae"
        ] = mean_absolute_error(
            true_train_targets[:, target_index],
            train_predictions[:, target_index]
        )

        result[
            f"{target}_valid_mae"
        ] = mean_absolute_error(
            true_valid_targets[:, target_index],
            valid_predictions[:, target_index]
        )

        result[
            f"{target}_train_rmse"
        ] = np.sqrt(
            mean_squared_error(
                true_train_targets[:, target_index],
                train_predictions[:, target_index]
            )
        )

        result[
            f"{target}_valid_rmse"
        ] = np.sqrt(
            mean_squared_error(
                true_valid_targets[:, target_index],
                valid_predictions[:, target_index]
            )
        )

        result[
            f"{target}_train_r2"
        ] = r2_score(
            true_train_targets[:, target_index],
            train_predictions[:, target_index]
        )

        result[
            f"{target}_valid_r2"
        ] = r2_score(
            true_valid_targets[:, target_index],
            valid_predictions[:, target_index]
        )

        valid_rmse_values.append(
            result[
                f"{target}_valid_rmse"
            ]
        )

        valid_r2_values.append(
            result[
                f"{target}_valid_r2"
            ]
        )

    result["mean_valid_rmse"] = np.mean(
        valid_rmse_values
    )

    result["mean_valid_r2"] = np.mean(
        valid_r2_values
    )

    architecture_evaluation_rows.append(
        result
    )

    del model
    gc.collect()
    torch.cuda.empty_cache()


architecture_evaluation = (
    pd.DataFrame(
        architecture_evaluation_rows
    )
    .sort_values(
        "hidden_neurons"
    )
    .reset_index(drop=True)
)

architecture_evaluation.to_csv(
    results_directory
    / "architecture_validation_metrics.csv",
    index=False
)

selected_architecture = (
    architecture_evaluation.loc[
        architecture_evaluation[
            "best_validation_loss"
        ].idxmin()
    ]
)

display(
    architecture_evaluation[
        [
            "hidden_neurons",
            "trainable_parameters",
            "best_epoch",
            "best_validation_loss",
            "jnk3_score_valid_mae",
            "jnk3_score_valid_rmse",
            "jnk3_score_valid_r2",
            "gsk3b_score_valid_mae",
            "gsk3b_score_valid_rmse",
            "gsk3b_score_valid_r2",
            "mean_valid_rmse",
            "mean_valid_r2"
        ]
    ]
)

print(
    "\nValidation-selected hidden size:",
    int(
        selected_architecture[
            "hidden_neurons"
        ]
    )
)

print(
    "Selected best epoch:",
    int(
        selected_architecture[
            "best_epoch"
        ]
    )
)

print(
    "Selected minimum validation loss:",
    round(
        selected_architecture[
            "best_validation_loss"
        ],
        6
    )
)

print(
    "\nSaved:",
    results_directory
    / "architecture_validation_metrics.csv"
)

In [ ]:
# Evaluate the validation-selected model on the test set

test_feature_file = Path(
    "mordred_test_features_filtered.pkl"
)

if not test_feature_file.exists():
    raise FileNotFoundError(
        f"Missing file: {test_feature_file}"
    )

# Load the test descriptors
X_test_all = pd.read_pickle(
    test_feature_file
).astype("float32")

missing_test_descriptors = [
    descriptor
    for descriptor in reduced_descriptors
    if descriptor not in X_test_all.columns
]

if missing_test_descriptors:
    raise ValueError(
        "Selected descriptors missing from "
        f"the test data: {missing_test_descriptors[:10]}"
    )

if not X_test_all.index.isin(
    model_data.index
).all():
    raise ValueError(
        "Test feature indices do not match "
        "model_dataset.csv."
    )

X_test_reduced = X_test_all[
    reduced_descriptors
].copy()

y_test = model_data.loc[
    X_test_reduced.index,
    target_columns
].astype("float32")

if X_test_reduced.isna().any().any():
    raise ValueError(
        "Missing values found in test features."
    )

if y_test.isna().any().any():
    raise ValueError(
        "Missing values found in test targets."
    )

# Apply transformations fitted on the training set
X_test_scaled = feature_scaler.transform(
    X_test_reduced
).astype("float32")

X_test_tensor = torch.from_numpy(
    X_test_scaled
)

selected_hidden_size = int(
    selected_architecture[
        "hidden_neurons"
    ]
)

selected_model_file = (
    results_directory
    / f"hidden_{selected_hidden_size}"
    / "best_model.pt"
)

selected_checkpoint = torch.load(
    selected_model_file,
    map_location="cpu",
    weights_only=False
)

selected_model = OneHiddenLayerRegressor(
    input_size=input_size,
    hidden_size=selected_hidden_size,
    output_size=output_size
).to(device)

selected_model.load_state_dict(
    selected_checkpoint[
        "model_state_dict"
    ]
)

test_predictions_scaled = predict_in_batches(
    selected_model,
    X_test_tensor
)

test_predictions = target_scaler.inverse_transform(
    test_predictions_scaled
)

true_test_targets = y_test[
    target_columns
].to_numpy()

final_metric_rows = []

for target_index, target in enumerate(
    target_columns
):

    final_metric_rows.append({
        "split": "Training",
        "target": target,
        "mae": selected_architecture[
            f"{target}_train_mae"
        ],
        "rmse": selected_architecture[
            f"{target}_train_rmse"
        ],
        "r2": selected_architecture[
            f"{target}_train_r2"
        ]
    })

    final_metric_rows.append({
        "split": "Validation",
        "target": target,
        "mae": selected_architecture[
            f"{target}_valid_mae"
        ],
        "rmse": selected_architecture[
            f"{target}_valid_rmse"
        ],
        "r2": selected_architecture[
            f"{target}_valid_r2"
        ]
    })

    final_metric_rows.append({
        "split": "Test",
        "target": target,
        "mae": mean_absolute_error(
            true_test_targets[:, target_index],
            test_predictions[:, target_index]
        ),
        "rmse": np.sqrt(
            mean_squared_error(
                true_test_targets[:, target_index],
                test_predictions[:, target_index]
            )
        ),
        "r2": r2_score(
            true_test_targets[:, target_index],
            test_predictions[:, target_index]
        )
    })

final_model_metrics = pd.DataFrame(
    final_metric_rows
)

test_predictions_table = pd.DataFrame(
    index=X_test_reduced.index
)

if "canonical_smiles" in model_data.columns:
    test_predictions_table[
        "canonical_smiles"
    ] = model_data.loc[
        X_test_reduced.index,
        "canonical_smiles"
    ]

for target_index, target in enumerate(
    target_columns
):

    test_predictions_table[
        f"actual_{target}"
    ] = true_test_targets[:, target_index]

    test_predictions_table[
        f"predicted_{target}"
    ] = test_predictions[:, target_index]

final_model_metrics.to_csv(
    results_directory
    / "selected_model_final_metrics.csv",
    index=False
)

test_predictions_table.to_csv(
    results_directory
    / "selected_model_test_predictions.csv",
    index=False
)

display(final_model_metrics)

print(
    "\nSelected hidden neurons:",
    selected_hidden_size
)

print(
    "Selected trainable parameters:",
    selected_checkpoint[
        "trainable_parameters"
    ]
)

print(
    "Selected best epoch:",
    selected_checkpoint[
        "best_epoch"
    ]
)

print(
    "Test compounds:",
    len(X_test_reduced)
)

print("\nSaved:")
print(
    results_directory
    / "selected_model_final_metrics.csv"
)

print(
    results_directory
    / "selected_model_test_predictions.csv"
)

In [ ]:
# Neural-network capacity and validation performance

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Saved results folder
results_directory = Path("nn_10000_epochs_results")

# Load completed architecture results
architecture_evaluation = pd.read_csv(
    results_directory / "architecture_validation_metrics.csv"
)

capacity_plot_data = (
    architecture_evaluation[
        [
            "hidden_neurons",
            "trainable_parameters",
            "best_validation_loss"
        ]
    ]
    .sort_values("trainable_parameters")
    .reset_index(drop=True)
)

# Select architecture with lowest validation loss
selected_architecture = architecture_evaluation.loc[
    architecture_evaluation["best_validation_loss"].idxmin()
]

# Number of training compounds
number_of_training_samples = 67502

# Dissertation purple + blue palette
blue = "#3A8EC1"
light_blue = "#8FC3E3"
purple = "#B784E8"
dark_purple = "#7A4FA3"
grid_colour = "#D9D9D9"
text_colour = "#1F1F1F"

fig, ax = plt.subplots(figsize=(8.2, 5.8))

# Capacity curve
ax.plot(
    capacity_plot_data["trainable_parameters"],
    capacity_plot_data["best_validation_loss"],
    marker="o",
    markersize=7.5,
    linewidth=1.8,
    color=blue,
    label="Neural-network architectures"
)

# Label each point by hidden-layer width
for _, row in capacity_plot_data.iterrows():
    ax.annotate(
        f'{int(row["hidden_neurons"])}',
        (
            row["trainable_parameters"],
            row["best_validation_loss"]
        ),
        xytext=(0, 8),
        textcoords="offset points",
        ha="center",
        fontsize=9,
        color=text_colour
    )

# Highlight selected model
selected_parameters = int(
    selected_architecture["trainable_parameters"]
)

selected_loss = float(
    selected_architecture["best_validation_loss"]
)

ax.scatter(
    selected_parameters,
    selected_loss,
    s=180,
    marker="D",
    color=purple,
    edgecolor=dark_purple,
    linewidth=1.3,
    zorder=4,
    label="Selected 2,000-neuron model"
)

# Parameter-to-sample reference lines
ax.axvline(
    number_of_training_samples,
    linestyle="--",
    linewidth=1.3,
    color=light_blue,
    label="Training samples"
)

ax.axvline(
    2 * number_of_training_samples,
    linestyle=":",
    linewidth=1.5,
    color=dark_purple,
    label="2 × training samples"
)

ax.set_xscale("log")

ax.set_xlabel(
    "Number of trainable parameters (log scale)",
    fontsize=12,
    color=text_colour
)

ax.set_ylabel(
    "Minimum validation MSE",
    fontsize=12,
    color=text_colour
)

ax.set_title(
    "Neural-network capacity and validation performance",
    fontsize=13,
    pad=12,
    color=text_colour
)

ax.grid(
    axis="both",
    linestyle="--",
    linewidth=0.7,
    color=grid_colour,
    alpha=0.6
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.tick_params(
    axis="both",
    labelsize=10.5,
    colors=text_colour
)

ax.legend(
    frameon=False,
    fontsize=9.5,
    loc="upper right"
)

fig.tight_layout()

fig.savefig(
    results_directory / "figure_4A_nn_capacity.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

print(
    "Selected hidden neurons:",
    int(selected_architecture["hidden_neurons"])
)

print(
    "Minimum validation loss:",
    round(selected_loss, 6)
)

print(
    "Saved:",
    results_directory / "figure_4A_nn_capacity.png"
)

In [ ]:
# Training and validation loss with early-epoch inset

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# Load saved results

results_directory = Path("nn_10000_epochs_results")

architecture_evaluation = pd.read_csv(
    results_directory / "architecture_validation_metrics.csv"
)

selected_architecture = architecture_evaluation.loc[
    architecture_evaluation["best_validation_loss"].idxmin()
]

selected_hidden_size = int(
    selected_architecture["hidden_neurons"]
)

selected_best_epoch = int(
    selected_architecture["best_epoch"]
)

selected_best_loss = float(
    selected_architecture["best_validation_loss"]
)

selected_history = pd.read_csv(
    results_directory
    / f"hidden_{selected_hidden_size}"
    / "training_history.csv"
)

# Main figure

fig, ax = plt.subplots(figsize=(8.5, 6.0))

# Training loss
ax.plot(
    selected_history["epoch"],
    selected_history["training_loss"],
    color=blue,
    linewidth=1.4,
    label="Training loss"
)

# Validation loss
ax.plot(
    selected_history["epoch"],
    selected_history["validation_loss"],
    color=purple,
    linewidth=1.4,
    label="Validation loss"
)

# Selected checkpoint
ax.axvline(
    selected_best_epoch,
    color=dark_purple,
    linestyle="--",
    linewidth=1.4,
    label=f"Selected checkpoint (epoch {selected_best_epoch})"
)

# Minimum validation-loss point
ax.scatter(
    selected_best_epoch,
    selected_best_loss,
    s=115,
    marker="D",
    color=purple,
    edgecolor=dark_purple,
    linewidth=1.2,
    zorder=5
)

ax.set_xlabel(
    "Training epoch",
    fontsize=12,
    color=text_colour
)

ax.set_ylabel(
    "Standardised mean squared error",
    fontsize=12,
    color=text_colour
)

ax.set_title(
    "Training and validation loss of the selected 2,000-neuron network",
    fontsize=13,
    pad=12,
    color=text_colour
)

ax.grid(
    axis="both",
    linestyle="--",
    linewidth=0.7,
    color=grid_colour,
    alpha=0.6
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.tick_params(
    axis="both",
    labelsize=10.5,
    colors=text_colour
)

ax.legend(
    frameon=False,
    fontsize=9.5,
    loc="upper left"
)

# Inset: first 200 epochs

early_history = selected_history[
    selected_history["epoch"] <= 200
].copy()

axins = inset_axes(
    ax,
    width="39%",
    height="39%",
    loc="center right",
    borderpad=1.6
)

axins.plot(
    early_history["epoch"],
    early_history["training_loss"],
    color=blue,
    linewidth=1.2
)

axins.plot(
    early_history["epoch"],
    early_history["validation_loss"],
    color=purple,
    linewidth=1.2
)

axins.axvline(
    selected_best_epoch,
    color=dark_purple,
    linestyle="--",
    linewidth=1.1
)

axins.scatter(
    selected_best_epoch,
    selected_best_loss,
    s=65,
    marker="D",
    color=purple,
    edgecolor=dark_purple,
    linewidth=1.0,
    zorder=5
)

axins.annotate(
    "Epoch 30",
    xy=(
        selected_best_epoch,
        selected_best_loss
    ),
    xytext=(62, 0.215),
    fontsize=8.5,
    color=dark_purple,
    arrowprops=dict(
        arrowstyle="-",
        color=dark_purple,
        linewidth=0.8
    )
)

axins.set_xlim(0, 200)

axins.set_title(
    "Early training (epochs 1–200)",
    fontsize=9.5,
    pad=5
)

axins.tick_params(
    axis="both",
    labelsize=8
)

axins.grid(
    linestyle="--",
    linewidth=0.5,
    color=grid_colour,
    alpha=0.6
)

for spine in axins.spines.values():
    spine.set_color("#777777")
    spine.set_linewidth(0.8)

# Layout

fig.subplots_adjust(
    left=0.12,
    right=0.98,
    bottom=0.12,
    top=0.90
)

# Save

figure_file = (
    results_directory
    / "figure_4B_nn_training_history.png"
)

fig.savefig(
    figure_file,
    dpi=600,
    bbox_inches="tight"
)

plt.show()

print("Selected hidden neurons:", selected_hidden_size)
print("Selected checkpoint epoch:", selected_best_epoch)

print(
    "Minimum validation loss:",
    round(selected_best_loss, 6)
)

print(
    "Validation loss at epoch 10,000:",
    round(
        float(
            selected_history.iloc[-1]["validation_loss"]
        ),
        6
    )
)

print(
    "Saved:",
    figure_file
)

In [ ]:
import pandas as pd

pred_file = (
    "nn_10000_epochs_results/"
    "selected_model_test_predictions.csv"
)

df = pd.read_csv(pred_file)

print(df.columns.tolist())
print()
display(df.head())
print()
print("Shape:", df.shape)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from matplotlib.lines import Line2D

# Load saved scaffold-test predictions
pred_file = "nn_10000_epochs_results/selected_model_test_predictions.csv"
df = pd.read_csv(pred_file)


def prepare_target(df, actual_col, pred_col):
    temp = df.copy()
    temp["actual"] = temp[actual_col]
    temp["pred"] = temp[pred_col]
    temp["abs_error"] = np.abs(temp["actual"] - temp["pred"])

    best = temp.nsmallest(3, "abs_error").copy().reset_index(drop=True)
    worst = temp.nlargest(3, "abs_error").copy().reset_index(drop=True)

    r2 = r2_score(temp["actual"], temp["pred"])
    mae = mean_absolute_error(temp["actual"], temp["pred"])
    rmse = np.sqrt(mean_squared_error(temp["actual"], temp["pred"]))

    return temp, best, worst, r2, mae, rmse


jnk, jnk_best, jnk_worst, jnk_r2, jnk_mae, jnk_rmse = prepare_target(
    df, "actual_jnk3_score", "predicted_jnk3_score"
)

gsk, gsk_best, gsk_worst, gsk_r2, gsk_mae, gsk_rmse = prepare_target(
    df, "actual_gsk3b_score", "predicted_gsk3b_score"
)

# Palette
BLUE = "#3A8EC1"
PURPLE = "#B784E8"
DARK_PURPLE = "#7A4FA3"
GREEN = "green"
RED = "darkred"
GRID = "#D9D9D9"
TEXT = "#1F1F1F"
GREY = "#8A8F98"

# Common axis limits, shared by both panels
all_values = np.concatenate([
    jnk["actual"].values,
    jnk["pred"].values,
    gsk["actual"].values,
    gsk["pred"].values
])

lower = np.floor(all_values.min()) - 0.3
upper = np.ceil(all_values.max()) + 0.3

hist_bins = np.linspace(lower, upper, 41)

# Figure geometry. Panel width is derived from panel height so the
# scatter axes is square in inches and the 1:1 line sits at 45 degrees.
fig_w = 13.0
fig_h = 7.2

panel_h = 0.52
panel_w = panel_h * fig_h / fig_w

bottom = 0.155
top_h = 0.095
right_w = 0.052
gap = 0.012

fig = plt.figure(figsize=(fig_w, fig_h), facecolor="white")

fig.suptitle(
    "Observed versus predicted docking scores on the scaffold test set",
    fontsize=13.5,
    color=TEXT,
    y=0.965
)

# Fixed label slots in axes coordinates so callout boxes can never collide
best_slots = [(0.045, 0.63), (0.045, 0.53), (0.045, 0.43)]
worst_slots = [(0.955, 0.34), (0.955, 0.23), (0.955, 0.12)]


def place_labels(ax, frame, slots, prefix, colour, align):
    # Assign slots top to bottom by predicted value so leader lines do not cross
    order = np.argsort(-frame["pred"].values)

    for slot_index, row_index in enumerate(order):
        row = frame.iloc[row_index]
        ax.annotate(
            f"{prefix} {row_index + 1}",
            xy=(row["actual"], row["pred"]),
            xytext=slots[slot_index],
            textcoords=ax.transAxes,
            fontsize=8,
            color=colour,
            ha=align,
            va="center",
            zorder=8,
            bbox=dict(boxstyle="round,pad=0.22", fc="white", ec=colour, lw=0.8),
            arrowprops=dict(arrowstyle="-", color=colour, lw=0.7,
                            shrinkA=2, shrinkB=3, alpha=0.9)
        )


def make_panel(left, data, best, worst, target_name, base_colour,
               r2, mae, rmse, panel_letter):
    ax = fig.add_axes([left, bottom, panel_w, panel_h])
    ax_top = fig.add_axes([left, bottom + panel_h + gap, panel_w, top_h], sharex=ax)
    ax_right = fig.add_axes([left + panel_w + gap, bottom, right_w, panel_h], sharey=ax)

    ax.set_axisbelow(True)

    ax.grid(True, color=GRID, linewidth=0.6)

    ax.scatter(
        data["actual"],
        data["pred"],
        s=6,
        alpha=0.18,
        color=base_colour,
        edgecolors="none",
        rasterized=True,
        zorder=2
    )

    ax.plot(
        [lower, upper],
        [lower, upper],
        linestyle=(0, (4, 3)),
        color=GREY,
        linewidth=1.2,
        zorder=3
    )

    ax.scatter(best["actual"], best["pred"], s=38, color=GREEN,
               edgecolor="white", linewidth=0.8, zorder=6)

    ax.scatter(worst["actual"], worst["pred"], s=38, color=RED,
               edgecolor="white", linewidth=0.8, zorder=6)

    place_labels(ax, best, best_slots, "Best", GREEN, "left")
    place_labels(ax, worst, worst_slots, "Worst", RED, "right")

    # Metrics block
    ax.text(
        0.035, 0.965,
        rf"$R^2$ = {r2:.3f}",
        transform=ax.transAxes,
        fontsize=9,
        color=TEXT,
        va="top"
    )

    ax.text(
        0.035, 0.912,
        f"MAE {mae:.2f}   RMSE {rmse:.2f} kcal mol$^{{-1}}$",
        transform=ax.transAxes,
        fontsize=8,
        color="#5A5F66",
        va="top"
    )

    ax.set_xlim(lower, upper)
    ax.set_ylim(lower, upper)
    ax.set_aspect("equal", adjustable="box")

    ax.set_xlabel(f"Observed {target_name} docking score (kcal mol$^{{-1}}$)",
                  fontsize=10, color=TEXT)
    ax.set_ylabel(f"Predicted {target_name} docking score (kcal mol$^{{-1}}$)",
                  fontsize=10, color=TEXT)

    ax.tick_params(axis="both", labelsize=8.5, colors="#4E5560", length=3)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#A9ADB3")
    ax.spines["bottom"].set_color("#A9ADB3")

    # Marginal distributions, shared bin edges so panels are comparable
    ax_top.hist(data["actual"], bins=hist_bins, color=base_colour,
                alpha=0.8, edgecolor="white", linewidth=0.25)

    ax_right.hist(data["pred"], bins=hist_bins, orientation="horizontal",
                  color=base_colour, alpha=0.8, edgecolor="white", linewidth=0.25)

    ax_top.axis("off")
    ax_right.axis("off")

    heading_y = bottom + panel_h + gap + top_h + 0.03

    fig.text(left - 0.042, heading_y, panel_letter,
             fontsize=11.5, fontweight="bold", color=TEXT)

    fig.text(left, heading_y, target_name,
             fontsize=11.5, fontweight="bold", color=TEXT)


make_panel(
    left=0.085,
    data=jnk,
    best=jnk_best,
    worst=jnk_worst,
    target_name="JNK3",
    base_colour=BLUE,
    r2=jnk_r2,
    mae=jnk_mae,
    rmse=jnk_rmse,
    panel_letter="A"
)

make_panel(
    left=0.575,
    data=gsk,
    best=gsk_best,
    worst=gsk_worst,
    target_name="GSK3\u03b2",
    base_colour=PURPLE,
    r2=gsk_r2,
    mae=gsk_mae,
    rmse=gsk_rmse,
    panel_letter="B"
)

legend_handles = [
    Line2D([0], [0], marker="o", linestyle="None", markerfacecolor=GREEN,
           markeredgecolor="white", markersize=6.5, label="Best predictions"),
    Line2D([0], [0], marker="o", linestyle="None", markerfacecolor=RED,
           markeredgecolor="white", markersize=6.5, label="Worst predictions"),
    Line2D([0], [0], color=GREY, linestyle=(0, (4, 3)), linewidth=1.3,
           label="Ideal (1:1)")
]

fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=3,
    frameon=False,
    fontsize=9,
    bbox_to_anchor=(0.5, 0.045),
    handletextpad=0.6,
    columnspacing=1.8
)

png_file = "figure_observed_vs_predicted_best_worst_PPT_final.png"
pdf_file = "figure_observed_vs_predicted_best_worst_final.pdf"

plt.savefig(png_file, dpi=600, facecolor="white")
plt.savefig(pdf_file, facecolor="white")

plt.show()

print("Saved:", png_file)
print("Saved:", pdf_file)